In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

# print("Path to dataset files:", path)

In [1]:
import os
import librosa
import librosa.display
from librosa.feature import melspectrogram
import matplotlib.pyplot as plt
import numpy as np
from torchvision.models import resnet18
from torch import nn

current_dir: str = os.getcwd()
ravdess_path = os.path.join(os.getcwd(), "..", "data", "ravdess")

In [6]:
class RAVDESSDataset:
    def __init__(self, base_path):
        self.base_path = base_path

    def load_audio(self, num_actors: int = 24):
        """
        load each audio (end with .wav) in each actor folder (start with Actor)
        save audio and audio data

        Args:
            num_actors (int, optional): The number of actors to load audio from. Defaults to 24.

        Returns:
            audio_files (list): A list of file paths for the loaded audio files.
            audio_data (list): A list of tuples containing the audio data and sample rate for each loaded audio file.
        """
        audio_files = []
        audio_data = []
        actor_folders = []

        all_actors = sorted(
            [d for d in os.listdir(self.base_path) if d.startswith("Actor")]
        )

        all_actors = all_actors[:num_actors]

        for actor in all_actors:
            actor_path = os.path.join(self.base_path, actor)

            for audio in os.listdir(actor_path):
                if audio.endswith(".wav"):
                    audio_files.append(os.path.join(actor_path, audio))
                    y, sr = librosa.load(os.path.join(actor_path, audio), sr=22050)
                    audio_data.append((y, sr))
            actor_folders.append(actor)

        return audio_files, audio_data, actor_folders

    def convert_to_spectrogram(self, audio_data: list, target_time: int = 128):
        """
            Convert the loaded audio data into mel spectrograms, normalize them, and pad or truncate them to a specified target time.

        Args:
            audio_data (list): A list of tuples containing the audio data and sample rate for each loaded audio file, which will be processed to create mel spectrograms.
            target_time (int, optional): The target time (in frames) to which the spectrograms will be padded or truncated. Defaults to 128.

        Returns:
            spectrograms (list): A list of processed mel spectrograms corresponding to the loaded audio data, ready for use in further analysis or model training.
        """
        spectrograms_raw = []
        spectrograms_norm = []

        for y, sr in audio_data:
            S = melspectrogram(y=y, sr=sr, n_mels=128)
            S_dB = librosa.power_to_db(S, ref=np.max)

            # Pad / truncate
            time = S_dB.shape[1]
            if time >= target_time:
                start = (time - target_time) // 2
                S_dB = S_dB[:, start : start + target_time]
            else:
                pad_left = (target_time - time) // 2
                pad_right = target_time - time - pad_left
                S_dB = np.pad(S_dB, ((0, 0), (pad_left, pad_right)))

            # dùng để lưu ảnh
            spectrograms_raw.append(S_dB)  

            # Normalize sau
            S_norm = (S_dB - np.mean(S_dB)) / (np.std(S_dB) + 1e-6)

            # dùng để train
            spectrograms_norm.append(S_norm)  

        return spectrograms_raw, spectrograms_norm

    def create_spectrogram_result(
        self, spectrograms: list, audio_files: list, result_folder: str, sr
    ):
        """
            Create a folder to save the spectrogram images and save each spectrogram as an image file.

        Args:
            spectrograms (list): A list of spectrograms to be saved as images.
            audio_files (list):  A list of file paths for the loaded audio files, used to determine the naming and organization of the saved spectrogram images.
            result_folder (str): The path to the folder where the spectrogram images will be saved.
            sr (int, optional): The sample rate to be used when saving the spectrogram images.
        """
        if not os.path.exists(result_folder):
            os.makedirs(result_folder)

        for spectrogram, audio_file in zip(spectrograms, audio_files):
            actor_name = os.path.basename(os.path.dirname(audio_file))
            if not os.path.exists(os.path.join(result_folder, actor_name)):
                os.makedirs(os.path.join(result_folder, actor_name))
            result_path = os.path.join(result_folder, actor_name)
            spectrogram_filename = (
                os.path.splitext(os.path.basename(audio_file))[0] + "_spectrogram.png"
            )
            spectrogram_filepath = os.path.join(result_path, spectrogram_filename)

            self.save_mel_spectrogram(spectrogram, spectrogram_filepath, sr=sr)

            print(
                f"Saved spectrogram for {os.path.basename(spectrogram_filename)} at {actor_name}"
            )

    def save_mel_spectrogram(self, spectrogram, spectrogram_filepath, sr):
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(spectrogram, sr=sr, x_axis="time", y_axis="mel")
        plt.colorbar(format="%+2.0f dB")
        plt.title("Mel Spectrogram")
        plt.tight_layout()
        plt.savefig(spectrogram_filepath)
        plt.close()

    def __len__(self):
        return len(os.listdir(self.base_path))
    
    def get_target_time(self, audio_data: list):
        max_time = 0
        for y, sr in audio_data:
            S = melspectrogram(y=y, sr=sr, n_mels=128)
            S_dB = librosa.power_to_db(S, ref=np.max)
            time = S_dB.shape[1]
            if time > max_time:
                max_time = time
        return max_time

**Load audio files, audio data**

In [7]:
dataset = RAVDESSDataset(ravdess_path)

audio_files, audio_data, actor_folders = dataset.load_audio()

target_time = dataset.get_target_time(audio_data)

spectrograms_raw, spectrograms_norm = dataset.convert_to_spectrogram(audio_data, target_time)


print(f"Sample rate: {audio_data[0][1]}")

Sample rate: 22050


**Convert dataset to mel spectrogram**

In [ ]:
dataset.create_spectrogram_result(spectrograms_raw, audio_files, result_folder="spectrograms", sr=22050)

Saved spectrogram for 03-01-01-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-03-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-03-01-01-02-01_spectrogram.png at Actor_01
Saved 

In [ ]:
def ResNet18(input_shape=(3, 224, 224), num_classes=8):
    model = resnet18(pretrained=True)

    old_weights = model.conv1.weight.data

    model.conv1 = nn.Conv2d(
        input_shape[0], 64, kernel_size=7, stride=2, padding=3, bias=False
    )

    model.conv1.weight.data = old_weights.mean(dim=1, keepdim=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    return model